# Pocket Duelist — fine-tune Qwen2.5-3B (QLoRA) → GGUF q4_k_m
Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
The finished file is saved to Google Drive: `MyDrive/pocket_duelist/qwen2.5-3b-instruct.Q4_K_M.gguf`.
Copy it to `C:\pokemon\models\` on your PC.

In [ ]:
# 1) Mount Google Drive first (so the result survives a disconnect)
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/pocket_duelist'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 2) Install Unsloth
!pip install -q unsloth

In [ ]:
# 3) Load base model in 4-bit + add LoRA adapters
from unsloth import FastLanguageModel
MAX_SEQ = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32, lora_alpha=32, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

In [ ]:
# 4) Dataset: 1,616 ChatML scenarios from the GitHub repo
from datasets import load_dataset
DATA_URL = 'https://raw.githubusercontent.com/chokunkie02/pokemon/main/backend/data/sft_data.jsonl'
split = load_dataset('json', data_files=DATA_URL, split='train').train_test_split(test_size=0.08, seed=3407)
test_rows = split['test']   # held out: never trained on
ds = split['train']
ds = ds.map(lambda b: {'text': [tokenizer.apply_chat_template(m, tokenize=False) for m in b['messages']]},
            batched=True, remove_columns=ds.column_names)
print(len(ds)); print(ds[0]['text'][:500])

In [ ]:
# 5) Train: 3 epochs, lr 2e-4, cosine (loss only on the assistant JSON)
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=3, learning_rate=2e-4, warmup_steps=20,
        lr_scheduler_type='cosine', optim='adamw_8bit', weight_decay=0.01,
        logging_steps=10, seed=3407, output_dir='outputs', report_to='none',
    ),
)
trainer = train_on_responses_only(trainer,
    instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
trainer.train()

In [ ]:
# 6) Accuracy on held-out examples the model never saw
import json
FastLanguageModel.for_inference(model)
valid = same_type = exact = 0
for r in test_rows:
    msgs = r['messages']
    prompt = tokenizer.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=128, do_sample=False)
    text = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    tgt = json.loads(msgs[2]['content'])
    try:
        got = json.loads(text); valid += 1
    except Exception:
        continue
    if got.get('type') == tgt.get('type'):
        same_type += 1
        exact += got == tgt
n = len(test_rows)
print(f'held-out {n} | valid JSON {valid}/{n} | same action type {same_type}/{n} | exact {exact}/{n}')

In [ ]:
# 7) Merge + export GGUF q4_k_m, copy to Drive
import glob, shutil
for old in glob.glob('/content/gguf_out*'):
    shutil.rmtree(old, ignore_errors=True)   # never pick up a file from an earlier run
model.save_pretrained_gguf('gguf_out', tokenizer, quantization_method='q4_k_m')
ggufs = [p for p in glob.glob('/content/gguf_out*/**/*.gguf', recursive=True) if 'q4_k_m' in p.lower()]
print(ggufs)
dst = os.path.join(OUT_DIR, 'qwen2.5-3b-instruct.Q4_K_M.gguf')
shutil.copy(ggufs[0], dst)
print('DONE ->', dst, round(os.path.getsize(dst)/1e9, 2), 'GB')